# 🌾 Market Management Database Explorer
This notebook provides an easy way to view all database tables and their contents directly from VS Code.

## Prerequisites
- AWS RDS PostgreSQL database is running
- Environment variables are set in `.env` file
- Required Python packages are installed

## 1. Install Required Libraries
Install and import necessary libraries for database connectivity and data visualization.

In [ ]:
# Install required packages if not already installed
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Uncomment if you need to install packages
# install_package("pandas")
# install_package("sqlalchemy")
# install_package("psycopg2-binary")
# install_package("python-dotenv")

print("✅ Packages ready!")

In [ ]:
# Import Required Libraries
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import create_engine, text
import psycopg2
from dotenv import load_dotenv
import os
from datetime import datetime
import json

print("✅ Libraries imported successfully!")

## 2. Connect to Database
Establish connection to your AWS RDS PostgreSQL database using credentials from environment file.

In [ ]:
# Load environment variables
load_dotenv()

# Get database credentials
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "postgres")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

print(f"🏠 Host: {DB_HOST}")
print(f"🔌 Port: {DB_PORT}")
print(f"🗃️  Database: {DB_NAME}")
print(f"👤 User: {DB_USER}")

# Create connection string
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create SQLAlchemy engine
engine = create_engine(connection_string, echo=False)

print("\n✅ Database connection established!")

## 3. List All Tables
Query the database metadata to retrieve and display all available tables.

In [ ]:
# Get list of all tables
query = """
SELECT table_name, 
       (SELECT COUNT(*) FROM information_schema.columns 
        WHERE table_name = t.table_name AND table_schema = 'public') as column_count
FROM information_schema.tables t
WHERE table_schema = 'public' 
ORDER BY table_name;
"""

tables_df = pd.read_sql(query, engine)
print("🗄️  DATABASE TABLES")
print("=" * 40)
print(tables_df.to_string(index=False))
print(f"\nTotal tables: {len(tables_df)}")

# Store table names for later use
table_names = tables_df['table_name'].tolist()
print(f"\nTable names: {table_names}")

## 4. Display Table Schema
Show table structure including column names, data types, and constraints for each table.

In [ ]:
# Function to get table schema
def get_table_schema(table_name):
    query = f"""
    SELECT column_name, data_type, is_nullable, column_default
    FROM information_schema.columns
    WHERE table_name = '{table_name}' AND table_schema = 'public'
    ORDER BY ordinal_position;
    """
    return pd.read_sql(query, engine)

# Display schema for all tables
for table_name in table_names:
    print(f"\n📋 SCHEMA: {table_name.upper()}")
    print("-" * 50)
    schema_df = get_table_schema(table_name)
    print(schema_df.to_string(index=False))
    print("\n")

## 5. Query Table Contents
Execute SELECT queries to retrieve data from specific tables with row counts and sample data.

In [ ]:
# Function to get table row count
def get_row_count(table_name):
    query = f"SELECT COUNT(*) as count FROM {table_name}"
    result = pd.read_sql(query, engine)
    return result.iloc[0]['count']

# Function to get sample data
def get_sample_data(table_name, limit=5):
    query = f"SELECT * FROM {table_name} LIMIT {limit}"
    return pd.read_sql(query, engine)

# Show row counts for all tables
print("📊 TABLE ROW COUNTS")
print("=" * 30)
row_counts = {}
for table_name in table_names:
    count = get_row_count(table_name)
    row_counts[table_name] = count
    print(f"{table_name:20} {count:>8} rows")

print(f"\nTotal rows across all tables: {sum(row_counts.values())}")

## 6. Display Results as DataFrames
Convert query results to pandas DataFrames for better visualization and manipulation.

In [ ]:
# Display sample data from each table with data
for table_name in table_names:
    count = row_counts[table_name]
    
    print(f"\n📊 TABLE: {table_name.upper()} ({count} rows)")
    print("=" * 60)
    
    if count > 0:
        sample_df = get_sample_data(table_name, 5)
        print(sample_df.to_string(index=False))
        
        if count > 5:
            print(f"\n... and {count - 5} more rows")
    else:
        print("No data in this table yet.")
    
    print("\n")

In [ ]:
# Interactive query function
def run_query(sql_query):
    """
    Run a custom SQL query and return results as DataFrame
    """
    try:
        result = pd.read_sql(sql_query, engine)
        return result
    except Exception as e:
        print(f"Error executing query: {e}")
        return None

# Example: Query crops table
print("📋 EXAMPLE: All Crops")
crops_df = run_query("SELECT * FROM crops ORDER BY category, name")
if crops_df is not None:
    print(crops_df.to_string(index=False))
else:
    print("No crops data available")

## 7. Export Table Data
Save table contents to various formats like CSV, Excel, or JSON files for further analysis.

In [ ]:
# Function to export table data
def export_table_data(table_name, format='csv'):
    """
    Export table data to file
    Supported formats: 'csv', 'json', 'excel'
    """
    query = f"SELECT * FROM {table_name}"
    df = pd.read_sql(query, engine)
    
    if len(df) == 0:
        print(f"No data to export from {table_name}")
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if format == 'csv':
        filename = f"exports/{table_name}_{timestamp}.csv"
        os.makedirs('exports', exist_ok=True)
        df.to_csv(filename, index=False)
        print(f"✅ Exported {table_name} to {filename}")
    
    elif format == 'json':
        filename = f"exports/{table_name}_{timestamp}.json"
        os.makedirs('exports', exist_ok=True)
        df.to_json(filename, orient='records', indent=2, default=str)
        print(f"✅ Exported {table_name} to {filename}")
    
    elif format == 'excel':
        filename = f"exports/{table_name}_{timestamp}.xlsx"
        os.makedirs('exports', exist_ok=True)
        df.to_excel(filename, index=False)
        print(f"✅ Exported {table_name} to {filename}")

# Example: Export crops data (if any)
if row_counts['crops'] > 0:
    export_table_data('crops', 'csv')
    export_table_data('crops', 'json')
else:
    print("No crops data to export")

In [ ]:
# Quick database summary
print("🌾 DATABASE SUMMARY")
print("=" * 40)
print(f"Database: {DB_NAME}")
print(f"Host: {DB_HOST}")
print(f"Total Tables: {len(table_names)}")
print(f"Total Records: {sum(row_counts.values())}")
print("\nTable Details:")
for table, count in row_counts.items():
    status = "📊 Has data" if count > 0 else "📭 Empty"
    print(f"  {table:20} {count:>6} rows  {status}")

print("\n✅ Database exploration complete!")
print("\n🔧 Next steps:")
print("1. Add sample data to test your application")
print("2. Create users, listings, and bids")
print("3. Test transaction workflows")
print("4. Run your FastAPI server: uvicorn backend.src.main:app --reload")